# Adaptive Computation Scheduling for Data Centers
## CAISO DLAP_SDGE Electricity Market Analysis

**Research project — UIC Computer Science**  
**Advisor:** Professor Ian Kash

This notebook runs the complete reproducible pipeline:

1. Load and validate the 2024 CAISO price data.
2. Run the baseline, day-ahead, and perfect-foresight optimization scenarios.
3. Compare seasonal savings.
4. Analyze negative-price periods.
5. Calculate hourly price profiles in Pacific local time.
6. Generate and display all result charts.

The two daylight-saving transition dates are excluded from the daily 24-hour optimization because those local dates contain 23 or 25 hours. All descriptive annual statistics still use the complete dataset.

## Setup

Create and activate the virtual environment from the repository root, then install the requirements:

```bash
python -m venv .venv
# macOS/Linux: source .venv/bin/activate
# Windows: .venv\Scripts\Activate.ps1
pip install -r requirements.txt
```

Start Jupyter from the repository root with `jupyter lab`, open this notebook, and select the `.venv` Python kernel.

In [ ]:
from pathlib import Path

import pandas as pd
from IPython.display import Image, display

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data").exists() or not (PROJECT_ROOT / "src").exists():
    raise FileNotFoundError(
        "Start Jupyter from the repository root so data/ and src/ are available."
    )

RESULTS_DIR = PROJECT_ROOT / "results"
print(f"Project root: {PROJECT_ROOT}")

## 1. Load and inspect the market data

In [ ]:
from src.data_utils import load_day_ahead, load_real_time, load_optimization_data

day_ahead = load_day_ahead()
real_time = load_real_time()
_, real_time_hourly, complete_days = load_optimization_data()

summary = pd.DataFrame(
    {
        "dataset": ["Day ahead", "Real time"],
        "rows": [len(day_ahead), len(real_time)],
        "minimum_LMP": [day_ahead["LMP"].min(), real_time["LMP"].min()],
        "maximum_LMP": [day_ahead["LMP"].max(), real_time["LMP"].max()],
        "mean_LMP": [day_ahead["LMP"].mean(), real_time["LMP"].mean()],
    }
).round(2)

print(f"Complete 24-hour Pacific-time days used for optimization: {len(complete_days)}")
display(summary)
display(day_ahead[["Time", "Market", "Location", "LMP"]].head())

## 2. Main optimization

- **Scenario 1 — Baseline:** spread 48 MWh evenly across 24 hours.
- **Scenario 2 — Day Ahead:** shift flexible computation using day-ahead prices.
- **Scenario 4 — Perfect Foresight:** use full knowledge of day-ahead and real-time prices as an upper bound.

In [ ]:
from src.optimize import run_all as run_optimization

optimization_results = run_optimization()
optimization_df = pd.DataFrame(optimization_results)
display(optimization_df)

In [ ]:
result_30 = optimization_df.loc[optimization_df["flex"] == 30].iloc[0]
print("30% workload flexibility")
print(f"Day-ahead savings: {result_30['savings_da_pct']:.2f}%")
print(f"Perfect-foresight savings: {result_30['savings_perfect_pct']:.2f}%")
print(f"Annual day-ahead savings: ${result_30['annual_savings_da']:,.2f}")
print(f"Annual perfect-foresight savings: ${result_30['annual_savings_perfect']:,.2f}")

## 3. Seasonal analysis at 30% flexibility

In [ ]:
from src.seasonal_analysis import run_seasonal

seasonal_results = run_seasonal()
seasonal_df = pd.DataFrame(seasonal_results).T
seasonal_df.index.name = "season"
display(seasonal_df)

## 4. Negative-price analysis

In [ ]:
from src.negative_price_analysis import analyze_negative_prices

negative_results = analyze_negative_prices()
display(pd.DataFrame([negative_results["day_ahead"], negative_results["real_time"]], index=["Day ahead", "Real time"]))

## 5. Average hourly price profile

In [ ]:
from src.hourly_profile import analyze_hourly_profile

hourly_results = analyze_hourly_profile()
hourly_mean = pd.Series(hourly_results["overall"]["mean"], name="Average LMP ($/MWh)")
hourly_mean.index = hourly_mean.index.astype(int)
hourly_mean.index.name = "Pacific local hour"
display(hourly_mean.to_frame())
print(hourly_results["summary"])

## 6. Generate every chart

In [ ]:
from src.generate_plots import generate_all_plots

generate_all_plots()

In [ ]:
chart_names = [
    "savings_curve.png",
    "duck_curve_sample.png",
    "monthly_prices.png",
    "seasonal_savings.png",
    "hourly_profile_seasonal.png",
    "negative_price_hours.png",
    "price_distribution.png",
]

for chart_name in chart_names:
    print(chart_name)
    display(Image(filename=str(RESULTS_DIR / chart_name), width=850))

## 7. Generated result files

In [ ]:
generated_files = sorted(path.name for path in RESULTS_DIR.iterdir() if path.is_file())
pd.DataFrame({"result_file": generated_files})

## Reproduce from the terminal

The notebook uses the same functions as the command-line pipeline. To regenerate every output without opening Jupyter, run:

```bash
python run_all.py
```

All outputs are written to `results/`.